# Laboratorio #4 - Aprendizaje por Refuerzo
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272

- Link del repositorio: https://github.com/alee2602/LAB4-RL


## **Task 1**

Una empresa de logística de última milla está evaluando el uso de robots autónomos para la gestión interna de su almacén principal. El almacén se modela como una cuadrícula de 8 × 8con puntos de recogida, puntos de entrega, zonas de penalización por congestión, y obstáculos fijos. El equipo de ingeniería necesita comparar dos estrategias de aprendizaje antes de comprometer recursos en un sistema completo: una política conservadora que aprenda a navegar de forma segura durante el entrenamiento, y una política agresiva que busque la ruta óptima sin importar los riesgos durante la exploración. Su grupo ha sido contratado para implementar ambas estrategias usando SARSA y Q-Learning respectivamente, comparar su comportamiento empírico, y producir un dictamen técnico con recomendaciones concretas para la gerencia

Diseñen formalmente el MDP que representa el almacén. El diseño debe especificar:

**1. El espacio de estados y el espacio de acciones. Justifiquen cada decisión considerando la interfaz de Gymnasium: observation_space y action_space deben ser instancias de gymnasium.spaces.Discrete o gymnasium.spaces.Box según corresponda. Argumenten cuál es más apropiado para este dominio.**

En el presente contexto, lo más apropiado es gymnasium.spaces.Discrete para ambos espacios, no Box. Cada celda de la cuadrícula 8x8 se codifica como un entero único con `estado = fila * 8 + columna`, dando Discrete(64), porque el agente siempre ocupa una posición exacta y no hay noción continua de ubicación. El uso de box tendría sentido con coordenadas continuas o imágenes como observación, pero aquí solo añadiría complejidad innecesaria. Para las acciones, el agente se mueve en cuatro direcciones, lo que se representa con Discrete(4).

**2. La función de recompensa con al menos tres componentes: recompensa por entrega exitosa, penalización por zona de congestión, y penalización por paso. Justifiquen la magnitud relativa de cada componente y argumenten qué comportamiento indeseable produciría una ponderación incorrecta de alguno de ellos.**

- **Recompensa por entrega exitosa:** un valor grande y positivo, por ejemplo +70, otorgado solo al llegar al punto de entrega, para que domine claramente sobre las penalizaciones acumuladas en una trayectoria típica.

- **Penalización por zona de congestión:** un valor negativo moderado, por ejemplo -5, aplicado al estar sobre una celda de congestión, suficiente para desincentivar pero sin que las rutas alternativas sean incoherentes.

- **Penalización por paso:** un valor pequeño y negativo, por ejemplo -1, en cada transición, para incentivar el uso de rutas cortas.

En este caso, una mala calibración produce comportamientos indeseables. Si el paso pesa demasiado frente a la congestión, el agente cruzará zonas de riesgo con tal de ahorrar tiempo. Si la congestión pesa demasiado frente a la entrega, el agente empezará a dar vueltas o evitará completar la tarea. Si la entrega no domina lo suficiente, el agente puede minimizar penalizaciones sin nunca llegar a la meta.

**3. La condición de terminación del episodio: ¿cuándo termina un episodio? ¿Es apropiado tener un
límite máximo de pasos? Justifiquen.**

- El episodio termina cuando el agente llega al punto de entrega. También conviene agregar un límite máximo de pasos, por ejemplo 100, como truncamiento. Esto evita que una política casi aleatoria en las primeras fases pueda quedar atrapada en ciclos infinitos. En Gymnasium esto se distingue con `terminated` (llegó a la meta) y `truncated` (se acabó el tiempo), siendo de utilidad para posteriores análisis. 

**4. El diseño del mapa 8 × 8: ubiquen al menos dos zonas de congestión adyacentes a rutas de altarecompensa. Esta configuración específica es crítica para que la comparación entre SARSA y Q-Learning sea informativa. Expliquen por qué esa configuración espacial genera el comportamiento
diferencial esperado entre ambos algoritmos.**

Consideramos que conviene ubicar al menos dos zonas de congestión sobre o muy cerca del camino más corto entre recogida y entrega. Por ejemplo, con inicio en la esquina superior izquierda y entrega en la inferior derecha, una franja de congestión puede cruzar la diagonal natural, dejando una ruta alternativa más larga que la rodea.

Esta configuración es la que hace útil la comparación entre algoritmos. Q-Learning es off-policy y aprende el valor máximo del siguiente estado sin importar la acción exploratoria tomada, por lo que su política greedy final puede explotar el camino corto arriesgado si la recompensa neta lo justifica. SARSA es on-policy y actualiza según la acción que realmente tomará bajo su política, incluyendo exploración epsilon-greedy, lo que lo hace más sensible al riesgo de caer en congestión por una acción aleatoria y lo lleva a preferir rutas más conservadoras. Sin esta adyacencia entre congestión y ruta óptima, ambos algoritmos convergerían a políticas casi idénticas, por lo que no habría punto de comparación.

## **Task 2**